# Multi-Cancer Explainable Screening: improved training notebook
Same setup as the paper (MobileNetV2, 160x160, Adam, batch 64, 80/20 split, seed 42, one model per cancer type), with these changes:
1. **Two-stage training:** head first (backbone frozen), then optional fine-tuning of the top backbone layers (`FINE_TUNE`).
2. **Preprocessing lives inside the model**, so Grad-CAM and LIME receive raw 0-255 images.
3. **Grad-CAM rewritten** (the old version failed on all 8 cancer types).
4. **One consistent evaluation:** every number comes from the restored best weights on the validation split.
5. **All paper figures are produced from real predictions** and saved to `/kaggle/working/figs`.

Whatever this notebook outputs is what the paper must report. If `FINE_TUNE=True`, the paper's method text must say the top backbone layers were fine-tuned.

In [ ]:
import os, json, random
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import warnings; warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_ROOT = "/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer"
WORK = "/kaggle/working"
MODEL_DIR, FIG_DIR = f"{WORK}/models", f"{WORK}/figs"
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

IMG_SIZE = (160, 160); BATCH_SIZE = 64; VAL_SPLIT = 0.2
EPOCHS_HEAD = 8                       # stage 1: frozen backbone, lr 1e-3
FINE_TUNE = True                      # stage 2: unfreeze top layers
EPOCHS_FT, UNFREEZE_FROM, FT_LR = 12, 100, 5e-5
WEIGHTS = "imagenet"
AUTOTUNE = tf.data.AUTOTUNE
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
def get_datasets(cancer):
    d = os.path.join(DATA_ROOT, cancer)
    kw = dict(validation_split=VAL_SPLIT, seed=SEED, image_size=IMG_SIZE,
              batch_size=BATCH_SIZE, label_mode="categorical")
    tr = tf.keras.utils.image_dataset_from_directory(d, subset="training", **kw)
    va = tf.keras.utils.image_dataset_from_directory(d, subset="validation", **kw)
    names = tr.class_names
    return tr.cache().prefetch(AUTOTUNE), va.cache().prefetch(AUTOTUNE), names

def build_model(num_classes):
    base = MobileNetV2(include_top=False, weights=WEIGHTS, input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    aug = keras.Sequential([layers.RandomFlip("horizontal"),
                            layers.RandomRotation(0.05),
                            layers.RandomZoom(0.1)], name="aug")
    inp = keras.Input((*IMG_SIZE, 3))
    x = aug(inp)                                        # active in training only
    x = layers.Rescaling(1/127.5, offset=-1, name="prep")(x)   # same as MobileNetV2 preprocess_input
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="drop1")(x)
    x = layers.Dense(128, activation="relu", name="fc128")(x)
    x = layers.Dropout(0.2, name="drop2")(x)
    out = layers.Dense(num_classes, activation="softmax", name="probs")(x)
    return keras.Model(inp, out), base

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"])

def callbacks(patience):
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7)]

def train_cancer(train_ds, val_ds, num_classes, epochs_head=None, epochs_ft=None, fine_tune=None):
    epochs_head = EPOCHS_HEAD if epochs_head is None else epochs_head
    epochs_ft = EPOCHS_FT if epochs_ft is None else epochs_ft
    fine_tune = FINE_TUNE if fine_tune is None else fine_tune
    model, base = build_model(num_classes)
    compile_model(model, 1e-3)
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs_head, callbacks=callbacks(2), verbose=2)
    hist = {k: list(v) for k, v in h.history.items()}
    if fine_tune:
        base.trainable = True
        for l in base.layers[:UNFREEZE_FROM]: l.trainable = False
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization): l.trainable = False
        compile_model(model, FT_LR)                    # recompile after changing trainable flags
        h2 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft, callbacks=callbacks(3), verbose=2)
        for k, v in h2.history.items(): hist.setdefault(k, []).extend(v)
    return model, hist

def evaluate(model, val_ds, names):
    yt, yp = [], []
    for x, y in val_ds:
        yp.extend(np.argmax(model.predict(x, verbose=0), 1)); yt.extend(np.argmax(y.numpy(), 1))
    yt, yp = np.array(yt), np.array(yp)
    return dict(acc=float((yt == yp).mean()), classes=list(names),
                cm=confusion_matrix(yt, yp).tolist(),
                report=classification_report(yt, yp, target_names=names, output_dict=True))

In [ ]:
cancer_types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
print(cancer_types)
RES_PATH = f"{WORK}/results.json"
RESULTS = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}

for cancer in cancer_types:
    if cancer in RESULTS:
        print("Skipping (done):", cancer); continue
    print("=" * 60, "\n", cancer, "\n", "=" * 60)
    tr, va, names = get_datasets(cancer)
    model, hist = train_cancer(tr, va, len(names))
    r = evaluate(model, va, names); r["history"] = hist
    RESULTS[cancer] = r
    model.save(f"{MODEL_DIR}/{cancer.replace(' ', '_')}.keras")
    json.dump(RESULTS, open(RES_PATH, "w"))
    print(f"{cancer}: validation accuracy = {r['acc']*100:.2f}%")

summary = pd.DataFrame({"cancer": list(RESULTS), "val_acc_%": [round(RESULTS[c]["acc"]*100, 2) for c in RESULTS]})
print(summary.to_string(index=False))

In [ ]:
# Export what the web app needs: copy MODEL_DIR (models + class_names.json) into the project's models/ folder
meta = {"demo": False, "cancers": {c: {"classes": RESULTS[c]["classes"], "val_acc": RESULTS[c]["acc"]} for c in RESULTS}}
json.dump(meta, open(f"{MODEL_DIR}/class_names.json", "w"), indent=1)
print("Saved", f"{MODEL_DIR}/class_names.json")

## Paper figures (generated from the real results above)

In [ ]:
order = ["ALL", "Brain Cancer", "Breast Cancer", "Cervical Cancer", "Kidney Cancer", "Lung and Colon Cancer", "Lymphoma", "Oral Cancer"]
order = [c for c in order if c in RESULTS]
short = {"Brain Cancer": "Brain", "Breast Cancer": "Breast", "Cervical Cancer": "Cervical", "Kidney Cancer": "Kidney",
         "Lung and Colon Cancer": "Lung &\nColon", "Oral Cancer": "Oral"}

# Fig. 4: final validation accuracy
fig, ax = plt.subplots(figsize=(8, 3.5))
acc = [RESULTS[c]["acc"] for c in order]
bars = ax.bar([short.get(c, c) for c in order], acc, color=sns.color_palette("tab10", len(order)))
for b, a in zip(bars, acc): ax.text(b.get_x()+b.get_width()/2, a+0.002, f"{a*100:.1f}%", ha="center", fontsize=8)
ax.set_ylim(max(0.5, min(acc)-0.05), 1.02); ax.set_ylabel("Validation accuracy"); ax.set_title("Final Validation Accuracy by Cancer-Type Model")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig4_val_accuracy.png", dpi=200); plt.show()

# Fig. 5: training curves (oral, brain)
sel = [c for c in ["Oral Cancer", "Brain Cancer"] if c in RESULTS]
fig, axes = plt.subplots(len(sel), 2, figsize=(9, 3.4*len(sel)), squeeze=False)
for i, c in enumerate(sel):
    h = RESULTS[c]["history"]
    for j, (k, t) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
        axes[i, j].plot(range(1, len(h[k])+1), h[k], label="Train"); axes[i, j].plot(range(1, len(h["val_"+k])+1), h["val_"+k], label="Val")
        axes[i, j].set_title(f"{c} - {t}"); axes[i, j].set_xlabel("Epoch"); axes[i, j].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig5_curves.png", dpi=200); plt.show()

# Fig. 6: normalised confusion matrices (oral, brain)
fig, axes = plt.subplots(1, len(sel), figsize=(5.5*len(sel), 4.5), squeeze=False)
for ax, c in zip(axes[0], sel):
    cm = np.array(RESULTS[c]["cm"], float); cm /= cm.sum(1, keepdims=True); n = [s.split("_")[-1] for s in RESULTS[c]["classes"]]
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=n, yticklabels=n, ax=ax)
    ax.set_title(f"Confusion Matrix - {c}"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig6_confusion.png", dpi=200); plt.show()

# Fig. 7: precision / recall / F1 for the same classes
rows = []
for c in sel:
    for k in RESULTS[c]["classes"]:
        r = RESULTS[c]["report"][k]; rows.append((f"{k}\n({c.split()[0]})", r["precision"], r["recall"], r["f1-score"]))
df = pd.DataFrame(rows, columns=["cls", "Precision", "Recall", "F1-Score"]).set_index("cls")
df.plot.bar(figsize=(9, 3.8), rot=0); plt.ylim(max(0, df.values.min()-0.05), 1.02); plt.title("Precision, Recall and F1-Score")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig7_prf.png", dpi=200); plt.show()
print(df.round(3))

# Fig. 8: accuracy vs number of subclasses
x = [len(RESULTS[c]["classes"]) for c in order]; y = acc
plt.figure(figsize=(5, 3.8)); plt.scatter(x, y, s=60)
for c, xi, yi in zip(order, x, y): plt.annotate(short.get(c, c).replace("\n", " "), (xi, yi), textcoords="offset points", xytext=(5, 4), fontsize=8)
if len(set(x)) > 1: z = np.polyfit(x, y, 1); xs = np.linspace(min(x)-.3, max(x)+.3, 10); plt.plot(xs, np.polyval(z, xs), "--", c="gray")
plt.xlabel("Number of subclasses"); plt.ylabel("Validation accuracy"); plt.title("Accuracy vs. Task Granularity")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig8_granularity.png", dpi=200); plt.show()

## Grad-CAM (fixed) and LIME

In [ ]:
def gradcam(model, img, class_idx=None):
    # img: raw 0-255 array (H, W, 3). Works because the backbone is split from the head by hand.
    base = next(l for l in model.layers if l.name.startswith("mobilenetv2"))
    conv_model = keras.Model(base.inputs, [base.get_layer("out_relu").output, base.output])
    head = [model.get_layer(n) for n in ("gap", "drop1", "fc128", "drop2", "probs")]
    x = preprocess_input(tf.convert_to_tensor(img[None].astype("float32")))
    with tf.GradientTape() as tape:
        conv, feat = conv_model(x, training=False)
        y = feat
        for l in head: y = l(y, training=False)
        if class_idx is None: class_idx = int(tf.argmax(y[0]))
        score = y[:, class_idx]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    return cam / (cam.max() + 1e-12), class_idx

def overlay(img, cam, alpha=0.4):
    h = cv2.resize(cam, (img.shape[1], img.shape[0]))
    h = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*h), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return np.uint8(h*alpha + img*(1-alpha))

def sample_image(cancer, class_name, correct_only=True, model=None):
    _, va, names = get_datasets(cancer); ci = names.index(class_name)
    for x, y in va:
        for k in range(len(x)):
            if np.argmax(y[k].numpy()) != ci: continue
            img = x[k].numpy()
            if correct_only and np.argmax(model.predict(img[None], verbose=0)) != ci: continue
            return img, names
    raise RuntimeError("no suitable image")

In [ ]:
%pip install -q lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

def load(cancer): return keras.models.load_model(f"{MODEL_DIR}/{cancer.replace(' ', '_')}.keras")

# Grad-CAM for every cancer type (one correctly classified example each)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, cancer in zip(axes.ravel(), order):
    m = load(cancer); names = RESULTS[cancer]["classes"]
    img, _ = sample_image(cancer, names[0], True, m)
    cam, ci = gradcam(m, img); ax.imshow(overlay(np.uint8(img), cam)); ax.set_title(f"{cancer}\n{names[ci]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/gradcam_all.png", dpi=200); plt.show()

# Fig. 10(a): Grad-CAM on a glioma image
m = load("Brain Cancer"); img, names = sample_image("Brain Cancer", "brain_glioma", True, m)
cam, ci = gradcam(m, img)
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
ax[0].imshow(np.uint8(img)); ax[0].set_title("Input Slice (Glioma)")
ax[1].imshow(cam, cmap="jet"); ax[1].set_title("Grad-CAM Heatmap")
ax[2].imshow(overlay(np.uint8(img), cam)); ax[2].set_title("Overlay")
[a.axis("off") for a in ax]; plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig10a_gradcam_glioma.png", dpi=200); plt.show()

# Fig. 10(b): LIME on an OSCC patch (raw 0-255 images go straight into the model)
m = load("Oral Cancer"); img, names = sample_image("Oral Cancer", "oral_scc", True, m)
exp = lime_image.LimeImageExplainer().explain_instance(
    np.uint8(img).astype("double"), lambda b: m.predict(b.astype("float32"), verbose=0),
    top_labels=2, hide_color=0, num_samples=1000, batch_size=64)
tmp, mask = exp.get_image_and_mask(exp.top_labels[0], positive_only=True, num_features=5, hide_rest=False)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(np.uint8(img)); ax[0].set_title("Input Patch (OSCC)")
ax[1].imshow(mark_boundaries(tmp/255.0, mask)); ax[1].set_title("LIME (superpixels, positive weight)")
[a.axis("off") for a in ax]; plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig10b_lime_oscc.png", dpi=200); plt.show()

## Before you update the paper
- Compare `results.json` with Fig. 4 and the oral/brain numbers in Section IV. Report what the notebook produced.
- Zip and download `/kaggle/working/figs` for the paper's figures.
- If `FINE_TUNE=True`, edit the abstract, Section III-C and III-E so they say the top backbone layers were fine-tuned (stage 2: lr 5e-5).
- The dataset contains augmented copies of images; mention this as a limitation of the random 80/20 split.